# AgriVision — Eksperimen v4b (LANJUTAN): komposisi data latih dan arsitektur (lab → lapangan)
### Opsi 2 (ablasi komposisi data) + Opsi 3 (modifikasi/perbandingan arsitektur) pada protokol bebas kebocoran

Hasil v3/v3b: pra-pemrosesan tidak menaikkan akurasi lapangan, dan model berbasis lab (±95%) turun ke ±50–57% di PlantDoc dan ±36–50% di Tomato-Village (di bawah tebakan mayoritas 56%). Notebook ini menguji **apa yang sebenarnya menutup kesenjangan itu**:

- **Bagian 1 — komposisi data latih** (MobileNetV2, tanpa pra-pemrosesan): `A_lab` (lab saja), `B_lab_pd` (lab + PlantDoc), `C_lab_tv` (lab + Tomato-Village), `D_lab_pd_tv` (lab + keduanya). Porsi data lapangan dalam batch dijaga sama (25%) agar yang diuji adalah *sumber*, bukan *jumlah*.
- **Bagian 2 — arsitektur** pada komposisi `B_lab_pd`, diuji di Tomato-Village (eksternal): MobileNetV2 + SE, MobileNetV2 + CBAM, MobileNetV3-Large, EfficientNetB0 (dibanding MobileNetV2 dari Bagian 1).

**Set evaluasi:** `lab_test` (PlantVillage, per daun), `field_test` (PlantDoc), `tv_test` (split *test* Tomato-Village setelah buang near-duplicate; dapat dibandingkan lintas semua kondisi), `tv_all` (seluruh Tomato-Village, 1.616 gambar — hanya sah untuk kondisi yang TIDAK melatih dengan Tomato-Village: A dan B). Tomato-Village hanya mencakup 3 kelas (Early_blight, Late_blight, healthy).

**Menjalankan:** Runtime → Run all (T4). Tiap model ~10–13 menit; Bagian 1 = 12 model, Bagian 2 = 12 model. Kuota GPU gratis Colab terbatas: urutan eksekusi berbasis seed (semua konfigurasi seed 42 dulu, lalu seed 1, dst.) sehingga hasil parsial tetap seimbang. **Setiap model mencetak satu baris `RESULT|{json}`** — jika runtime terputus, baris-baris itu cukup untuk merekonstruksi tabel.

**Notebook lanjutan:** v4 penuh terhenti setelah 9 dari 24 model (kuota GPU habis). Hasil 9 model itu tersimpan di `ml-service/results/v4_partial_results.csv`. Notebook ini hanya menjalankan yang belum ada: Bagian 1 = B, C, D seed 2 (3 model); Bagian 2 = 4 arsitektur × seed 42, 1, 2 (12 model). Tabel ringkasan di Step 15 hanya memuat model yang dilatih pada sesi ini; gabungkan dengan CSV di atas untuk tabel lengkap.

In [ ]:
# Cek GPU aktif — kalau kosong/error: Runtime > Change runtime type > T4 GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Step 0 — Install & import

In [ ]:
!pip install -q datasets scikit-learn seaborn
import os, re, json, math, shutil, random, hashlib
import numpy as np
import tensorflow as tf
from PIL import Image

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU tersedia:", tf.config.list_physical_devices("GPU"))

## Step 1 — Kelas kanonik (10 kelas tomat)

Urutan **harus identik** dengan v1 / `ml-service/models/label_map.json` (indeks 0–9), supaya `disease_reference` dan service tidak perlu berubah.

In [ ]:
TOMATO_CLASSES = [
    "Tomato___Bacterial_spot",
    "Tomato___Early_blight",
    "Tomato___Late_blight",
    "Tomato___Leaf_Mold",
    "Tomato___Septoria_leaf_spot",
    "Tomato___Spider_mites Two-spotted_spider_mite",
    "Tomato___Target_Spot",
    "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
    "Tomato___Tomato_mosaic_virus",
    "Tomato___healthy",
]
assert TOMATO_CLASSES == sorted(TOMATO_CLASSES), "urutan harus alfabetis (sama dengan v1)"

def norm(s):
    return re.sub(r"[^a-z0-9]", "", s.lower())

CANON_BY_NORM = {norm(c): c for c in TOMATO_CLASSES}

DATA_DIR = "/content/data_v2"
SPLITS = ["lab_train", "lab_val", "lab_test", "field_train", "field_val", "field_test"]
shutil.rmtree(DATA_DIR, ignore_errors=True)
for sp in SPLITS:
    for c in TOMATO_CLASSES:
        os.makedirs(f"{DATA_DIR}/{sp}/{c}", exist_ok=True)

# registry: split -> list of (path, class_name, dhash)
records = {sp: [] for sp in SPLITS}

def dhash(img, size=8):
    g = img.convert("L").resize((size + 1, size), Image.LANCZOS)
    a = np.asarray(g, dtype=np.int16)
    bits = (a[:, 1:] > a[:, :-1]).flatten()
    return int("".join("1" if b else "0" for b in bits), 2)

def save_image(img, split, cls, idx):
    path = f"{DATA_DIR}/{split}/{cls}/{split}_{idx:06d}.jpg"
    img.convert("RGB").save(path, quality=95)
    return path

## Step 2 — PlantVillage (lab) → `lab_train` / `lab_val` / `lab_test`

Split asli dataset (per daun) dipakai untuk `lab_test`. `lab_val` diambil dari train **dengan `GroupShuffleSplit` per `leaf_id`**, lalu dicek tidak ada `leaf_id` yang bocor antar set.

In [ ]:
# mohanty/PlantVillage adalah dataset berbasis skrip (plant_village.py) yang TIDAK bisa dimuat dengan
# load_dataset() di versi `datasets` terbaru (BuilderConfig 'color' not found). Jadi file mentahnya dibaca
# langsung: data.zip (raw/color/<kelas>/<file>), splits/color_{train,test}.txt, leaf_grouping/leaf-map.json.
from huggingface_hub import hf_hub_download
import zipfile
from sklearn.model_selection import GroupShuffleSplit

REPO = "mohanty/PlantVillage"
zip_path = hf_hub_download(REPO, "data.zip", repo_type="dataset")   # +-2,2 GB
leaf_map = json.load(open(hf_hub_download(REPO, "leaf_grouping/leaf-map.json", repo_type="dataset"), encoding="utf-8"))
split_lines = {
    s: open(hf_hub_download(REPO, f"splits/color_{s}.txt", repo_type="dataset"), encoding="utf-8").read().splitlines()
    for s in ["train", "test"]
}
print("contoh baris split:", split_lines["train"][:2])

zf = zipfile.ZipFile(zip_path)
member_by_rel = {}
for n in zf.namelist():
    if "raw/color/Tomato" in n and not n.endswith("/"):
        member_by_rel[n[n.index("raw/color/"):]] = n
print("file tomat berwarna di zip:", len(member_by_rel), "| contoh:", list(member_by_rel)[:2])
assert member_by_rel, "tidak ada raw/color/Tomato* di data.zip — cek struktur: zf.namelist()[:10]"

def leaf_id_for(filename, class_name):
    # meniru turunan leaf_id di plant_village.py (hasil kira-kira; lihat pengecekan kebocoran di bawah)
    name = filename.replace("_final_masked", "")
    if "___" in name:
        name = name.split("___")[-1]
    low = name.lower()
    if "copy" in low:
        name = name[: low.index("copy")]
    for ext in (".jpg", ".png"):
        if name.lower().endswith(ext):
            name = name[: -len(ext)]
    ident = name.strip().lower()
    sugg = leaf_map.get(ident)
    if isinstance(sugg, str):
        sugg = [sugg]
    if sugg:
        if len(sugg) == 1:
            return str(sugg[0])
        for sg in sugg:
            if class_name in str(sg):
                return str(sg)
    return f"fallback_{ident}"

def parse(lines):
    out = []
    for ln in lines:
        parts = ln.strip().split("/")
        if len(parts) < 4 or parts[1] != "color":
            continue
        if norm(parts[2]) not in CANON_BY_NORM:
            continue
        rel = "/".join(parts)
        if rel in member_by_rel:
            out.append((rel, CANON_BY_NORM[norm(parts[2])], leaf_id_for(parts[-1], parts[2])))
    return out

pv_train_items, pv_test_items = parse(split_lines["train"]), parse(split_lines["test"])
print("PlantVillage tomat — train:", len(pv_train_items), "test:", len(pv_test_items))
assert pv_train_items and pv_test_items, "parse() menghasilkan 0 baris — bandingkan contoh baris split dengan format yang diharapkan (raw/color/<kelas>/<file>)"
n_fb = sum(1 for it in pv_train_items + pv_test_items if it[2].startswith("fallback_"))
print(f"leaf_id fallback (tidak ada di leaf-map): {n_fb} dari {len(pv_train_items) + len(pv_test_items)}")

In [ ]:
# Pengecekan kebocoran per daun: buang dari train semua gambar yang leaf_id-nya juga ada di test
test_leaves = {it[2] for it in pv_test_items}
before = len(pv_train_items)
pv_train_items = [it for it in pv_train_items if it[2] not in test_leaves]
print("gambar train dibuang karena leaf_id ada di test:", before - len(pv_train_items))

# lab_val dipisah per leaf_id dari train
leaf_train = np.array([it[2] for it in pv_train_items])
gss = GroupShuffleSplit(n_splits=1, test_size=0.10, random_state=SEED)
tr_idx, va_idx = next(gss.split(np.zeros(len(leaf_train)), groups=leaf_train))
assert not (set(leaf_train[tr_idx]) & set(leaf_train[va_idx])), "ada leaf_id yang sama di train dan val"
assert not (set(leaf_train) & test_leaves), "ada leaf_id yang sama di train dan test"
print("OK — tidak ada leaf_id yang sama antara lab_train, lab_val, dan lab_test")

def dump(items, indices, split):
    for i, k in enumerate(indices):
        rel, cls, _ = items[int(k)]
        with zf.open(member_by_rel[rel]) as fh:
            img = Image.open(fh).convert("RGB")
        path = save_image(img, split, cls, i)
        records[split].append((path, cls, dhash(img)))

dump(pv_train_items, tr_idx, "lab_train")
dump(pv_train_items, va_idx, "lab_val")
dump(pv_test_items, range(len(pv_test_items)), "lab_test")
print({sp: len(records[sp]) for sp in ["lab_train", "lab_val", "lab_test"]})

## Step 3 — PlantDoc (non-lab) → `field_train` / `field_val` / `field_test`

Pemetaan kelas PlantDoc → label kanonik. Yang tidak bisa dipetakan **tidak dipakai**. Duplikat (hash gambar sama) dibuang sebelum split; gambar yang muncul di dua kelas berbeda dibuang seluruhnya.

In [ ]:
from datasets import load_dataset
from sklearn.model_selection import train_test_split

pdoc = load_dataset("Project-AgML/plant_doc_classification", split="train")
pdoc_names = pdoc.features["label"].names
print("Kelas PlantDoc:", len(pdoc_names))

PLANTDOC_TO_CANON = {
    "Tomato Early blight leaf": "Tomato___Early_blight",
    "Tomato Septoria leaf spot": "Tomato___Septoria_leaf_spot",
    "Tomato leaf": "Tomato___healthy",
    "Tomato leaf bacterial spot": "Tomato___Bacterial_spot",
    "Tomato leaf late blight": "Tomato___Late_blight",
    "Tomato leaf mosaic virus": "Tomato___Tomato_mosaic_virus",
    "Tomato leaf yellow virus": "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
    "Tomato mold leaf": "Tomato___Leaf_Mold",
    # "Tomato two spotted spider mites leaf": hanya 2 gambar -> dikecualikan (terlalu sedikit untuk di-split)
    # Target Spot: tidak ada di PlantDoc
}
missing = [k for k in PLANTDOC_TO_CANON if k not in pdoc_names]
assert not missing, f"nama kelas PlantDoc tidak ditemukan (nama berubah?): {missing}. Daftar: {pdoc_names}"
pdoc_idx_to_canon = {pdoc_names.index(k): v for k, v in PLANTDOC_TO_CANON.items()}

# kumpulkan gambar tomat PlantDoc (di memori; jumlahnya kecil, <1000)
pd_items = []
for ex in pdoc.filter(lambda lab: lab in pdoc_idx_to_canon, input_columns=["label"]):
    img = ex["image"].convert("RGB")
    pd_items.append((img, pdoc_idx_to_canon[ex["label"]], dhash(img)))
print("PlantDoc tomat (dipetakan):", len(pd_items))

# buang duplikat
by_hash = {}
for k, (_, cls, h) in enumerate(pd_items):
    by_hash.setdefault(h, []).append(k)
keep = []
dropped_dup = dropped_conflict = 0
for h, ks in by_hash.items():
    classes = {pd_items[k][1] for k in ks}
    if len(classes) > 1:
        dropped_conflict += len(ks)
    else:
        keep.append(ks[0]); dropped_dup += len(ks) - 1
print(f"duplikat dibuang: {dropped_dup}; gambar berlabel ganda dibuang: {dropped_conflict}; tersisa: {len(keep)}")

labels_keep = [pd_items[k][1] for k in keep]
# kelas dengan <4 gambar tidak bisa di-stratify 3 arah -> dikecualikan dengan peringatan
from collections import Counter
cnt = Counter(labels_keep)
too_small = {c for c, n in cnt.items() if n < 4}
if too_small:
    print("PERINGATAN kelas dikecualikan (terlalu sedikit):", too_small)
keep = [k for k in keep if pd_items[k][1] not in too_small]
labels_keep = [pd_items[k][1] for k in keep]

k_trval, k_test = train_test_split(keep, test_size=0.30, stratify=labels_keep, random_state=SEED)
k_train, k_val = train_test_split(k_trval, test_size=0.26, stratify=[pd_items[k][1] for k in k_trval], random_state=SEED)

for split, ks in [("field_train", k_train), ("field_val", k_val), ("field_test", k_test)]:
    for i, k in enumerate(ks):
        img, cls, h = pd_items[k]
        path = save_image(img, split, cls, i)
        records[split].append((path, cls, h))
print({sp: len(records[sp]) for sp in ["field_train", "field_val", "field_test"]})

## Step 4 — Anti-kebocoran: buang near-duplicate lintas set

Hash perseptual (dHash 64-bit). Gambar di set yang lebih "bersih" menjadi acuan: `field_test` dan `field_val` tidak boleh punya near-duplicate (jarak Hamming ≤ 3) di set latih.

In [ ]:
def hashes_of(split):
    return np.array([h for _, _, h in records[split]], dtype=np.uint64)

def popcount64(a):
    return np.unpackbits(np.ascontiguousarray(a).view(np.uint8).reshape(-1, 8), axis=1).sum(axis=1)

def purge(from_splits, against_split, thr=3):
    ref = hashes_of(against_split)
    removed = 0
    for sp in from_splits:
        kept = []
        for rec in records[sp]:
            h = np.uint64(rec[2])
            if len(ref) and (popcount64(ref ^ h).min() <= thr):
                os.remove(rec[0]); removed += 1
            else:
                kept.append(rec)
        records[sp] = kept
    return removed

r1 = purge(["lab_train", "lab_val", "lab_test", "field_train", "field_val"], "field_test")
r2 = purge(["lab_train", "field_train"], "field_val")
print("near-duplicate dibuang terhadap field_test:", r1, "| terhadap field_val:", r2)

## Step 5 — Audit dataset (jumlah per kelas per set)

In [ ]:
import pandas as pd

rows = []
for c in TOMATO_CLASSES:
    rows.append({"kelas": c.replace("Tomato___", "")} | {sp: sum(1 for r in records[sp] if r[1] == c) for sp in SPLITS})
audit = pd.DataFrame(rows).set_index("kelas")
audit.loc["TOTAL"] = audit.sum()
display(audit)
print("Kelas tanpa data lapangan (uji lapangan tidak mencakupnya):",
      [c for c in TOMATO_CLASSES if not any(r[1] == c for r in records["field_test"])])

## Step 6 — Modul pra-pemrosesan (diambil dari repo, identik dengan yang dipakai API)

In [ ]:
!rm -rf /content/agrivision && git clone -q https://github.com/Argy1/AgriVision.git /content/agrivision
import sys
sys.path.insert(0, "/content/agrivision/ml-service")
from app.ml.enhance import apply_variant, VARIANTS
print("Varian pra-pemrosesan:", list(VARIANTS))

## Step 7 — Bangun dataset per varian

Setiap gambar: buka → resize 224×224 (PIL bilinear, seperti di API) → `apply_variant` → simpan PNG (lossless). Dataset varian dibuat sekali lalu dipakai ulang untuk semua seed.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

IMG_SIZE = (224, 224)
VAR_ROOT = "/content/data_v3"

def _convert(args):
    src, dst, variant = args
    img = Image.open(src).convert("RGB").resize(IMG_SIZE, Image.BILINEAR)
    Image.fromarray(apply_variant(np.asarray(img, dtype=np.uint8), variant)).save(dst)

def build_variant(variant):
    root = f"{VAR_ROOT}/{variant}"
    if os.path.isdir(root):
        return root
    jobs = []
    for sp in SPLITS:
        for c in TOMATO_CLASSES:
            os.makedirs(f"{root}/{sp}/{c}", exist_ok=True)
        for path, cls, _h in records[sp]:
            jobs.append((path, f"{root}/{sp}/{cls}/{os.path.basename(path)[:-4]}.png", variant))
    with ThreadPoolExecutor(8) as ex:
        list(ex.map(_convert, jobs))
    return root

import time
t0 = time.time()
build_variant("none")
print(f"dataset 'none' siap ({time.time() - t0:.0f} dtk)")

## Step 8 — Pipeline data, model MobileNetV2, dan prosedur latih

Augmentasi (di GPU, di dalam model) dan semua hiperparameter identik untuk setiap varian. Tahap 1: kepala klasifikasi (Adam 1e-3, ≤8 epoch). Tahap 2: 60 lapisan atas dibuka kecuali BatchNorm (Adam 1e-4, ≤15 epoch). Early stopping memantau akurasi pada gabungan `lab_val` + `field_val`.

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

BATCH = 32
FIELD_WEIGHT = 0.25
AUTOTUNE = tf.data.AUTOTUNE

def make_augment():
    return tf.keras.Sequential([
        tf.keras.layers.RandomFlip("horizontal_and_vertical"),
        tf.keras.layers.RandomRotation(0.1),
        tf.keras.layers.RandomZoom(0.15),
        tf.keras.layers.RandomTranslation(0.05, 0.05),
        tf.keras.layers.RandomBrightness(0.15, value_range=(0, 255)),
        tf.keras.layers.RandomContrast(0.15),
    ], name="augment")

def load_dir_v(root, split, shuffle):
    return tf.keras.utils.image_dataset_from_directory(
        f"{root}/{split}", class_names=TOMATO_CLASSES, label_mode="int",
        image_size=IMG_SIZE, batch_size=None, shuffle=shuffle, seed=SEED,
    )

def to_u8(x, y):
    return tf.cast(x, tf.uint8), y

def to_f32(x, y):
    return tf.cast(x, tf.float32), y

def make_train_ds(root):
    lab = load_dir_v(root, "lab_train", True).map(to_u8).cache().shuffle(2000, seed=SEED).repeat()
    fld = load_dir_v(root, "field_train", True).map(to_u8).cache().shuffle(1000, seed=SEED).repeat()
    mixed = tf.data.Dataset.sample_from_datasets([lab, fld], weights=[1 - FIELD_WEIGHT, FIELD_WEIGHT], seed=SEED)
    return mixed.batch(BATCH).map(to_f32, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

def make_eval_ds(root, split):
    return load_dir_v(root, split, False).map(to_u8).cache().map(to_f32).batch(BATCH).prefetch(AUTOTUNE)

def make_val_ds(root):
    ds = load_dir_v(root, "lab_val", False).concatenate(load_dir_v(root, "field_val", False))
    return ds.map(to_u8).cache().map(to_f32).batch(BATCH).prefetch(AUTOTUNE)

n_lab, n_fld = len(records["lab_train"]), len(records["field_train"])
STEPS_PER_EPOCH = (n_lab + n_fld) // BATCH
counts = np.array([sum(1 for r in records["lab_train"] + records["field_train"] if r[1] == c) for c in TOMATO_CLASSES], dtype=float)
CLASS_WEIGHT = {i: float(counts.sum() / (len(counts) * max(n, 1))) for i, n in enumerate(counts)}

def build_mobilenet():
    inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
    x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)  # input mentah 0-255 -> [-1, 1] (di dalam graf)
    base = tf.keras.applications.MobileNetV2(input_shape=IMG_SIZE + (3,), include_top=False, weights="imagenet")
    base.trainable = False
    x = base(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(len(TOMATO_CLASSES), activation="softmax")(x)
    return tf.keras.Model(inputs, outputs, name="mobilenetv2_tomato"), base

def n_trainable(m):
    return int(sum(np.prod(w.shape) for w in m.trainable_weights))

def compile_model(model, lr):
    model.compile(optimizer=tf.keras.optimizers.Adam(lr), loss="sparse_categorical_crossentropy", metrics=["accuracy"])

def train_run(variant, seed):
    root = build_variant(variant)
    tf.keras.utils.set_random_seed(seed)
    core, base = build_mobilenet()
    model = tf.keras.Sequential([tf.keras.Input(shape=IMG_SIZE + (3,)), make_augment(), core], name="train_wrapper")
    train_ds, val_ds = make_train_ds(root), make_val_ds(root)
    stop = lambda p: [tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=p, restore_best_weights=True)]
    t0 = time.time()

    compile_model(model, 1e-3)
    trainable_stage1 = n_trainable(core)
    h1 = model.fit(train_ds, validation_data=val_ds, epochs=8, steps_per_epoch=STEPS_PER_EPOCH,
                   class_weight=CLASS_WEIGHT, callbacks=stop(3), verbose=0)

    base.trainable = True
    for layer in base.layers[:-60]:
        layer.trainable = False
    for layer in base.layers:
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False
    compile_model(model, 1e-4)
    trainable_stage2 = n_trainable(core)
    h2 = model.fit(train_ds, validation_data=val_ds, epochs=15, steps_per_epoch=STEPS_PER_EPOCH,
                   class_weight=CLASS_WEIGHT, callbacks=stop(4), verbose=0)
    info = {
        "epochs_stage1": len(h1.history["loss"]), "epochs_stage2": len(h2.history["loss"]),
        "best_val_acc": float(max(h1.history["val_accuracy"] + h2.history["val_accuracy"])),
        "train_acc_last": float(h2.history["accuracy"][-1]),
        "trainable_stage1": trainable_stage1, "trainable_stage2": trainable_stage2,
        "total_params": int(core.count_params()), "seconds": time.time() - t0,
    }
    return core, info, root

def predict_split(core, root, split):
    ds = make_eval_ds(root, split)
    y_true = np.concatenate([y.numpy() for _, y in ds])
    probs = core.predict(ds, verbose=0)
    return y_true, probs.argmax(1)

def run_metrics(y_true, y_pred):
    present = sorted(set(y_true.tolist()))
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, labels=present, zero_division=0)
    _, _, fw, _ = precision_recall_fscore_support(y_true, y_pred, labels=present, average="weighted", zero_division=0)
    return {"n": int(len(y_true)), "accuracy": float((y_true == y_pred).mean()),
            "macro_precision": float(p.mean()), "macro_recall": float(r.mean()),
            "macro_f1": float(f.mean()), "weighted_f1": float(fw)}

RESULTS = {}   # (variant, seed) -> {"info", "metrics": {split: {...}}, "preds": {split: (y_true, y_pred)}}

def run_all(variants, seeds):
    for v in variants:
        for sd in seeds:
            if (v, sd) in RESULTS:
                continue
            core, info, root = train_run(v, sd)
            preds = {sp: predict_split(core, root, sp) for sp in ["lab_val", "field_val", "lab_test", "field_test"]}
            metrics = {sp: run_metrics(*preds[sp]) for sp in preds}
            RESULTS[(v, sd)] = {"info": info, "metrics": metrics, "preds": preds}
            m = metrics
            print(f"[{v:13s} seed={sd}] ep={info['epochs_stage1']}+{info['epochs_stage2']} "
                  f"lab_test={m['lab_test']['accuracy']*100:.2f}% field_test={m['field_test']['accuracy']*100:.2f}% "
                  f"field_val={m['field_val']['accuracy']*100:.2f}% ({info['seconds']/60:.1f} mnt)")
            del core
            tf.keras.backend.clear_session()

## Step 9 — Ambil Tomato-Village (3 kelas) dan bersihkan near-duplicate

In [ ]:
import subprocess
from collections import Counter

TV_FOLDER_TO_CLASS = {"Early_blight": "Tomato___Early_blight", "Late_blight": "Tomato___Late_blight", "Healthy": "Tomato___healthy"}
TV_BASE = "/content/ext/tv/Variant-a(Multiclass Classification)"
sparse = " ".join(f'"Variant-a(Multiclass Classification)/{sp}/{f}"' for sp in ["train", "val", "test"] for f in TV_FOLDER_TO_CLASS)
cmds = f'''
set -e
rm -rf /content/ext && mkdir -p /content/ext && cd /content/ext
git clone -q --filter=blob:none --no-checkout --depth 1 https://github.com/mamta-joshi-gehlot/Tomato-Village.git tv
cd tv
git sparse-checkout init --cone
git sparse-checkout set {sparse}
git checkout -q
'''
subprocess.run(["bash", "-c", cmds], check=True)

def hash_path(p):
    return dhash(Image.open(p).convert("RGB"))

tv_raw = {}
for sp in ["train", "val", "test"]:
    items = []
    for folder, cls in TV_FOLDER_TO_CLASS.items():
        d = f"{TV_BASE}/{sp}/{folder}"
        for f in sorted(os.listdir(d)):
            if f.lower().endswith((".jpg", ".jpeg", ".png")):
                p = f"{d}/{f}"
                items.append((p, cls, hash_path(p)))
    tv_raw[sp] = items
print("Tomato-Village mentah:", {sp: len(v) for sp, v in tv_raw.items()})

def has_near(h, ref, thr=3):
    return len(ref) > 0 and popcount64(ref ^ np.uint64(h)).min() <= thr

# 1) buang yang near-duplicate terhadap data lab + PlantDoc (semua set)
lab_pd_hashes = np.array([h for sp in SPLITS for _, _, h in records[sp]], dtype=np.uint64)
tv_clean = {sp: [it for it in v if not has_near(it[2], lab_pd_hashes)] for sp, v in tv_raw.items()}
print("setelah buang near-dup vs lab/PlantDoc:", {sp: len(v) for sp, v in tv_clean.items()})

# 2) di dalam Tomato-Village: val vs train, test vs (train+val)
h_train = np.array([it[2] for it in tv_clean["train"]], dtype=np.uint64)
tv_clean["val"] = [it for it in tv_clean["val"] if not has_near(it[2], h_train)]
h_trval = np.array([it[2] for it in tv_clean["train"] + tv_clean["val"]], dtype=np.uint64)
tv_clean["test"] = [it for it in tv_clean["test"] if not has_near(it[2], h_trval)]
print("setelah buang near-dup internal:", {sp: len(v) for sp, v in tv_clean.items()})
tv_all_items = tv_clean["train"] + tv_clean["val"] + tv_clean["test"]
for sp in ["train", "val", "test"]:
    print(sp, dict(Counter(c.replace("Tomato___", "") for _, c, _ in tv_clean[sp])))

## Step 10 — Tulis split Tomato-Village ke direktori yang sama dengan data lain

In [ ]:
ROOT = f"{VAR_ROOT}/none"
build_variant("none")

def write_split(name, items):
    shutil.rmtree(f"{ROOT}/{name}", ignore_errors=True)
    for c in TOMATO_CLASSES:
        os.makedirs(f"{ROOT}/{name}/{c}", exist_ok=True)
    jobs = [(p, f"{ROOT}/{name}/{c}/{i:05d}.png", "none") for i, (p, c, _h) in enumerate(items)]
    with ThreadPoolExecutor(8) as ex:
        list(ex.map(_convert, jobs))

write_split("tv_train", tv_clean["train"])
write_split("tv_val", tv_clean["val"])
write_split("tv_test", tv_clean["test"])
write_split("tv_all", tv_all_items)

def class_counts(split):
    return np.array([len(os.listdir(f"{ROOT}/{split}/{c}")) for c in TOMATO_CLASSES], dtype=float)

def y_true_str(split):
    return "".join(str(i) * int(n) for i, n in enumerate(class_counts(split)))

SETS = ["lab_train", "field_train", "tv_train", "lab_val", "field_val", "tv_val", "lab_test", "field_test", "tv_test", "tv_all"]
display(pd.DataFrame({sp: class_counts(sp).astype(int) for sp in SETS}, index=[c.replace("Tomato___", "") for c in TOMATO_CLASSES]))

tv_test_counts = class_counts("tv_test")
print("Baris dasar tv_test (tebakan mayoritas):", round(tv_test_counts.max() / tv_test_counts.sum() * 100, 2), "% | n =", int(tv_test_counts.sum()))
tv_all_counts = class_counts("tv_all")
print("Baris dasar tv_all  (tebakan mayoritas):", round(tv_all_counts.max() / tv_all_counts.sum() * 100, 2), "% | n =", int(tv_all_counts.sum()))
print("YTRUE|field_test|" + y_true_str("field_test"))
print("YTRUE|tv_test|" + y_true_str("tv_test"))

## Step 11 — Komposisi data, model, dan arsitektur

Hiperparameter sama dengan v3 (augmentasi di GPU, Adam 1e-3 untuk kepala ≤8 epoch lalu 1e-4 untuk 60 lapisan atas dengan BatchNorm dibekukan ≤15 epoch, class weight, early stopping pada akurasi validasi gabungan dari sumber yang dipakai melatih). Langkah per epoch sama untuk semua kondisi.

`A_lab` memakai satu sumber; `B`/`C` memakai lab 75% + satu sumber lapangan 25%; `D` memakai lab 75% + PlantDoc 12,5% + Tomato-Village 12,5% (porsi lapangan total tetap 25%).

In [ ]:
TRAIN_DIR = {"lab": "lab_train", "pd": "field_train", "tv": "tv_train"}
VAL_DIR = {"lab": "lab_val", "pd": "field_val", "tv": "tv_val"}
CONDITIONS = {
    "A_lab": {"lab": 1.0},
    "B_lab_pd": {"lab": 0.75, "pd": 0.25},
    "C_lab_tv": {"lab": 0.75, "tv": 0.25},
    "D_lab_pd_tv": {"lab": 0.75, "pd": 0.125, "tv": 0.125},
}
EVAL_SETS = ["lab_test", "field_test", "tv_test", "tv_all"]
SAVE_PRED = ["field_test", "tv_test"]
STEPS_PER_EPOCH_V4 = STEPS_PER_EPOCH  # sama dengan v3 (lab+PlantDoc)

def make_mix_train_ds(cond):
    parts, weights = [], []
    for src, w in CONDITIONS[cond].items():
        ds = load_dir_v(ROOT, TRAIN_DIR[src], True).map(to_u8).cache().shuffle(2000 if src == "lab" else 1000, seed=SEED).repeat()
        parts.append(ds); weights.append(w)
    mixed = parts[0] if len(parts) == 1 else tf.data.Dataset.sample_from_datasets(parts, weights=weights, seed=SEED)
    return mixed.batch(BATCH).map(to_f32, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

def make_mix_val_ds(cond):
    ds = None
    for src in CONDITIONS[cond]:
        d = load_dir_v(ROOT, VAL_DIR[src], False)
        ds = d if ds is None else ds.concatenate(d)
    return ds.map(to_u8).cache().map(to_f32).batch(BATCH).prefetch(AUTOTUNE)

def class_weight_for(cond):
    counts = sum(class_counts(TRAIN_DIR[s]) for s in CONDITIONS[cond])
    return {i: float(counts.sum() / (len(counts) * max(n, 1))) for i, n in enumerate(counts)}

L = tf.keras.layers

class SpatialPool(L.Layer):
    # gabungan rata-rata dan maksimum sepanjang kanal (untuk perhatian spasial CBAM)
    def call(self, x):
        return tf.concat([tf.reduce_mean(x, axis=-1, keepdims=True), tf.reduce_max(x, axis=-1, keepdims=True)], axis=-1)
    def compute_output_shape(self, input_shape):
        return tuple(input_shape[:-1]) + (2,)

def se_block(x, ratio=16):
    c = x.shape[-1]
    s = L.GlobalAveragePooling2D(keepdims=True)(x)
    s = L.Dense(c // ratio, activation="relu")(s)
    s = L.Dense(c, activation="sigmoid")(s)
    return L.Multiply()([x, s])

def cbam_block(x, ratio=16):
    c = x.shape[-1]
    mlp1, mlp2 = L.Dense(c // ratio, activation="relu"), L.Dense(c)
    avg = mlp2(mlp1(L.GlobalAveragePooling2D(keepdims=True)(x)))
    mx = mlp2(mlp1(L.GlobalMaxPooling2D(keepdims=True)(x)))
    x = L.Multiply()([x, L.Activation("sigmoid")(L.Add()([avg, mx]))])
    sp = L.Conv2D(1, 7, padding="same", activation="sigmoid")(SpatialPool()(x))
    return L.Multiply()([x, sp])

def build_model(arch):
    inputs = tf.keras.Input(shape=IMG_SIZE + (3,))            # input mentah 0-255
    shape = IMG_SIZE + (3,)
    if arch in ("mobilenetv2", "mnv2_se", "mnv2_cbam"):
        x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
        base = tf.keras.applications.MobileNetV2(input_shape=shape, include_top=False, weights="imagenet")
    elif arch == "mobilenetv3large":
        x = inputs
        base = tf.keras.applications.MobileNetV3Large(input_shape=shape, include_top=False, weights="imagenet")  # preprocessing bawaan (input 0-255)
    elif arch == "efficientnetb0":
        x = inputs
        base = tf.keras.applications.EfficientNetB0(input_shape=shape, include_top=False, weights="imagenet")  # preprocessing bawaan (input 0-255)
    else:
        raise ValueError(arch)
    base.trainable = False
    x = base(x, training=False)
    if arch == "mnv2_se":
        x = se_block(x)
    elif arch == "mnv2_cbam":
        x = cbam_block(x)
    x = L.GlobalAveragePooling2D()(x)
    x = L.Dropout(0.3)(x)
    outputs = L.Dense(len(TOMATO_CLASSES), activation="softmax")(x)
    return tf.keras.Model(inputs, outputs, name=f"{arch}_tomato"), base

for a in ["mobilenetv2", "mnv2_se", "mnv2_cbam", "mobilenetv3large", "efficientnetb0"]:
    m, _ = build_model(a)
    print(f"{a:18s} total params: {m.count_params():>10,}")
    del m
tf.keras.backend.clear_session()

## Step 12 — Prosedur latih, evaluasi, dan pencetak `RESULT|` per model

In [ ]:
RESULTS = {}  # (arch, cond, seed) -> {"info", "metrics", "preds"}

def run_one(arch, cond, seed):
    key = (arch, cond, seed)
    if key in RESULTS:
        return
    tf.keras.utils.set_random_seed(seed)
    core, base = build_model(arch)
    model = tf.keras.Sequential([tf.keras.Input(shape=IMG_SIZE + (3,)), make_augment(), core], name="train_wrapper")
    train_ds, val_ds, cw = make_mix_train_ds(cond), make_mix_val_ds(cond), class_weight_for(cond)
    stop = lambda p: [tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=p, restore_best_weights=True)]
    t0 = time.time()
    compile_model(model, 1e-3)
    h1 = model.fit(train_ds, validation_data=val_ds, epochs=8, steps_per_epoch=STEPS_PER_EPOCH_V4, class_weight=cw, callbacks=stop(3), verbose=0)
    base.trainable = True
    for layer in base.layers[:-60]:
        layer.trainable = False
    for layer in base.layers:
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False
    compile_model(model, 1e-4)
    h2 = model.fit(train_ds, validation_data=val_ds, epochs=15, steps_per_epoch=STEPS_PER_EPOCH_V4, class_weight=cw, callbacks=stop(4), verbose=0)
    info = {"epochs": [len(h1.history["loss"]), len(h2.history["loss"])],
            "best_val_acc": float(max(h1.history["val_accuracy"] + h2.history["val_accuracy"])),
            "params": int(core.count_params()), "minutes": round((time.time() - t0) / 60, 1)}
    preds = {sp: predict_split(core, ROOT, sp) for sp in EVAL_SETS}
    metrics = {sp: run_metrics(*preds[sp]) for sp in preds}
    RESULTS[key] = {"info": info, "metrics": metrics, "preds": preds}
    line = {"arch": arch, "cond": cond, "seed": seed, "info": info,
            "acc": {sp: round(metrics[sp]["accuracy"], 4) for sp in EVAL_SETS},
            "f1": {sp: round(metrics[sp]["macro_f1"], 4) for sp in EVAL_SETS},
            "pred": {sp: "".join(map(str, preds[sp][1])) for sp in SAVE_PRED}}
    print("RESULT|" + json.dumps(line))
    print(f"[{arch:16s} {cond:12s} seed={seed}] lab={metrics['lab_test']['accuracy']*100:.2f} "
          f"PlantDoc={metrics['field_test']['accuracy']*100:.2f} TV_test={metrics['tv_test']['accuracy']*100:.2f} "
          f"TV_all={metrics['tv_all']['accuracy']*100:.2f} ({info['minutes']} mnt)")
    del core, model
    tf.keras.backend.clear_session()

def run_grid(configs, seeds):
    t0 = time.time()
    for sd in seeds:                       # seed di luar: hasil parsial tetap seimbang antar konfigurasi
        for arch, cond in configs:
            run_one(arch, cond, sd)
    print(f"Selesai dalam {(time.time() - t0) / 60:.0f} menit")

## Step 13 — BAGIAN 1: komposisi data latih (MobileNetV2, tanpa pra-pemrosesan)

In [ ]:
PART1 = [("mobilenetv2", c) for c in ["B_lab_pd", "C_lab_tv", "D_lab_pd_tv"]]   # A_lab seed 42/1/2, B/C/D seed 42/1 sudah ada (lihat ml-service/results/v4_partial_results.csv)
PART1_SEEDS = [2]
run_grid(PART1, PART1_SEEDS)

## Step 14 — BAGIAN 2: arsitektur pada komposisi `B_lab_pd` (diuji di Tomato-Village)

In [ ]:
PART2 = [(a, "B_lab_pd") for a in ["mnv2_se", "mnv2_cbam", "mobilenetv3large", "efficientnetb0"]]
PART2_SEEDS = [42, 1, 2]
run_grid(PART2, PART2_SEEDS)

## Step 15 — Ringkasan (rata-rata ± simpangan baku antarseed) dan uji McNemar berpasangan

In [ ]:
from scipy.stats import binomtest

def ms(vals):
    v = np.array(vals) * 100
    return f"{v.mean():.2f} ± {v.std():.2f}" if len(v) > 1 else f"{v.mean():.2f}"

def summary_table(keys_filter=None):
    rows = []
    groups = sorted({(a, c) for (a, c, s) in RESULTS if keys_filter is None or (a, c) in keys_filter})
    for a, c in groups:
        runs = [RESULTS[k] for k in RESULTS if k[0] == a and k[1] == c]
        row = {"arsitektur": a, "komposisi": c, "seed": len(runs), "params": f"{runs[0]['info']['params']:,}"}
        for sp in EVAL_SETS:
            row[f"{sp} acc"] = ms([r["metrics"][sp]["accuracy"] for r in runs])
        row["tv_test F1"] = ms([r["metrics"]["tv_test"]["macro_f1"] for r in runs])
        rows.append(row)
    return pd.DataFrame(rows).set_index(["arsitektur", "komposisi"])

print("tv_all hanya sah untuk komposisi tanpa Tomato-Village (A_lab, B_lab_pd). tv_test sah untuk semua.")
display(summary_table())

def mcnemar(y, pa, pb):
    a_ok, b_ok = pa == y, pb == y
    n10, n01 = int((a_ok & ~b_ok).sum()), int((~a_ok & b_ok).sum())
    return n10, n01, (binomtest(min(n10, n01), n10 + n01, 0.5).pvalue if n10 + n01 else 1.0)

def pairwise(ref, others, sets=("field_test", "tv_test")):
    rows = []
    for o in others:
        for sd in sorted({s for (_, _, s) in RESULTS}):
            if (*ref, sd) not in RESULTS or (*o, sd) not in RESULTS:
                continue
            for sp in sets:
                y, pa = RESULTS[(*ref, sd)]["preds"][sp]
                _, pb = RESULTS[(*o, sd)]["preds"][sp]
                n10, n01, p = mcnemar(y, pa, pb)
                rows.append({"pembanding": f"{ref[0]}/{ref[1]}", "uji": f"{o[0]}/{o[1]}", "seed": sd, "set": sp,
                             "selisih (poin)": f"{((pb == y).mean() - (pa == y).mean()) * 100:+.2f}", "ref benar/uji salah": n10, "ref salah/uji benar": n01, "p": f"{p:.4f}"})
    return pd.DataFrame(rows)

part1_others = [k for k in PART1 if k != ("mobilenetv2", "A_lab")]
print("\nBagian 1 — dibanding A_lab (lab saja):")
display(pairwise(("mobilenetv2", "A_lab"), part1_others))
part2_others = [k for k in PART2]
print("Bagian 2 — dibanding MobileNetV2/B_lab_pd:")
display(pairwise(("mobilenetv2", "B_lab_pd"), part2_others))

## Step 16 — Per kelas, ringkasan model, dan penyimpanan hasil

In [ ]:
from sklearn.metrics import classification_report
short = [c.replace("Tomato___", "") for c in TOMATO_CLASSES]
for key in [("mobilenetv2", "A_lab", 42), ("mobilenetv2", "B_lab_pd", 42), ("mobilenetv2", "D_lab_pd_tv", 42)]:
    if key not in RESULTS:
        continue
    for sp in ["field_test", "tv_test"]:
        y, p = RESULTS[key]["preds"][sp]
        present = sorted(set(y.tolist()))
        print(f"\n=== {key} — {sp} (n={len(y)}) ===")
        print(classification_report(y, p, labels=present, target_names=[short[i] for i in present], digits=4, zero_division=0))

m, _ = build_model("mobilenetv2")
m.summary()
del m

OUT4 = "/content/outputs_v4"
os.makedirs(OUT4, exist_ok=True)
out = {f"{k[0]}|{k[1]}|seed{k[2]}": {"info": v["info"], "metrics": v["metrics"]} for k, v in RESULTS.items()}
json.dump(out, open(f"{OUT4}/hasil_v4.json", "w", encoding="utf-8"), indent=2, ensure_ascii=False, default=float)
print("Tersimpan:", sorted(os.listdir(OUT4)))